# Model Serving with Snowpark Container Services

This notebook demonstrates **Snowflake Model Serving**:

- Deploy a model from the registry as a **real-time inference service**
- Call the model via **Python SDK**, **SQL**, or **HTTP endpoint**
- Manage service lifecycle (scale, suspend, resume)

The model runs on Snowpark Container Services (SPCS) with auto-scaling and a dedicated REST endpoint.

In [ ]:
import pandas as pd
import numpy as np
import json

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col
from snowflake.ml.registry import Registry

session = get_active_session()
session.query_tag = {"origin": "ml_deep_dive", "name": "banking_fraud_detection", "notebook": "04_model_serving"}

DB = "BANKING_ML_DEMO"
SCHEMA = "FRAUD_DETECTION"
WH = "ML_DEMO_WH"

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

print(f"Context: {session.get_fully_qualified_current_schema()}")

## Prerequisites

Before deploying, ensure:
1. Compute pool `ML_DEMO_CPU_POOL` exists (created in `sql/setup.sql`)
2. Role has `BIND SERVICE ENDPOINT` privilege
3. Model `FRAUD_DETECTION_MODEL` is registered (notebook 03)

In [ ]:
# Get model version from registry
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)
model = reg.get_model("FRAUD_DETECTION_MODEL")
mv = model.default  # V2_TUNED

print(f"Model: {model.name}")
print(f"Version: {mv.version_name}")
print(f"Comment: {mv.comment}")

## Deploy Model as a Service

One API call creates a fully managed inference endpoint on SPCS.

In [ ]:
# Deploy to Snowpark Container Services
SERVICE_NAME = "FRAUD_DETECTION_SERVICE"
COMPUTE_POOL = "ML_DEMO_CPU_POOL"

mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL,
    ingress_enabled=True,
    min_instances=1,
    max_instances=3
)

print(f"Service '{SERVICE_NAME}' deployed to compute pool '{COMPUTE_POOL}'")
print("Snowflake automatically builds the container image and starts the service.")

In [ ]:
# Check service status
services = mv.list_services()
print("=== Active Services ===")
print(services)

# Also check via SQL
session.sql(f"SHOW ENDPOINTS IN SERVICE {SERVICE_NAME}").show()

## Real-Time Inference via Python

Call the deployed model service directly from Python.

In [ ]:
# Create a sample transaction to score
sample_transaction = pd.DataFrame([{
    "AMOUNT": 5200.00,
    "IS_ONLINE": 1,
    "TXN_HOUR": 3,
    "TXN_DAY_OF_WEEK": 2,
    "TXN_IS_WEEKEND": 0,
    "TXN_IS_LATE_NIGHT": 1,
    "TXN_AMOUNT_RATIO": 8.5,
    "TXN_IS_HIGH_AMOUNT": 1,
    "CUST_AVG_TRANSACTION_AMT": 150.00,
    "CUST_TOTAL_TRANSACTIONS": 245,
    "CUST_STDDEV_AMOUNT": 120.00,
    "CUST_MAX_AMOUNT": 800.00,
    "CUST_UNIQUE_MERCHANTS": 35,
    "CUST_PCT_ONLINE": 0.3,
    "CUST_AGE": 35,
    "CUST_INCOME": 75000,
    "MERCH_AVG_TRANSACTION_AMT": 200.00,
    "MERCH_TOTAL_TRANSACTIONS": 1500,
    "MERCH_FRAUD_RATE": 0.01
}])

# Run prediction via the deployed SPCS service
prediction = mv.run(
    sample_transaction,
    function_name="predict",
    service_name=SERVICE_NAME
)
probability = mv.run(
    sample_transaction,
    function_name="predict_proba",
    service_name=SERVICE_NAME
)

print("=== Real-Time Inference Result ===")
print(f"Prediction: {'FRAUD' if prediction.iloc[0, 0] == 1 else 'LEGITIMATE'}")
print(f"Fraud probability: {probability.iloc[0, 1]:.4f}")
print(f"\nKey risk signals:")
print(f"  Amount: ${sample_transaction['AMOUNT'].values[0]:,.2f} (8.5x customer avg)")
print(f"  Time: 3 AM (late night)")
print(f"  Online transaction")

## Real-Time Inference via SQL

Call the model service function directly from SQL - great for batch scoring pipelines.

In [ ]:
# SQL-based inference using service functions
# The service exposes functions: SERVICE_NAME!PREDICT, SERVICE_NAME!PREDICT_PROBA

sql_query = f"""
SELECT 
    t.TRANSACTION_ID,
    t.AMOUNT,
    t.IS_FRAUD AS ACTUAL,
    {SERVICE_NAME}!PREDICT(
        t.AMOUNT, t.IS_ONLINE, HOUR(t.TIMESTAMP), DAYOFWEEK(t.TIMESTAMP),
        CASE WHEN DAYOFWEEK(t.TIMESTAMP) IN (0, 6) THEN 1 ELSE 0 END,
        CASE WHEN HOUR(t.TIMESTAMP) >= 22 OR HOUR(t.TIMESTAMP) <= 5 THEN 1 ELSE 0 END,
        t.AMOUNT / NULLIF(c.CUST_AVG_TRANSACTION_AMT, 0),
        CASE WHEN t.AMOUNT > 3 * c.CUST_AVG_TRANSACTION_AMT THEN 1 ELSE 0 END,
        c.CUST_AVG_TRANSACTION_AMT, c.CUST_TOTAL_TRANSACTIONS,
        c.CUST_STDDEV_AMOUNT, c.CUST_MAX_AMOUNT,
        c.CUST_UNIQUE_MERCHANTS, c.CUST_PCT_ONLINE,
        c.CUST_AGE, c.CUST_INCOME,
        m.MERCH_AVG_TRANSACTION_AMT, m.MERCH_TOTAL_TRANSACTIONS, m.MERCH_FRAUD_RATE
    ) AS FRAUD_PREDICTION
FROM RAW_TRANSACTIONS t
JOIN (
    SELECT CUSTOMER_ID, AVG(AMOUNT) AS CUST_AVG_TRANSACTION_AMT,
           COUNT(*) AS CUST_TOTAL_TRANSACTIONS, STDDEV(AMOUNT) AS CUST_STDDEV_AMOUNT,
           MAX(AMOUNT) AS CUST_MAX_AMOUNT, COUNT(DISTINCT MERCHANT_ID) AS CUST_UNIQUE_MERCHANTS,
           AVG(IS_ONLINE) AS CUST_PCT_ONLINE, AVG(CUSTOMER_AGE) AS CUST_AGE,
           AVG(CUSTOMER_INCOME) AS CUST_INCOME
    FROM RAW_TRANSACTIONS GROUP BY CUSTOMER_ID
) c ON t.CUSTOMER_ID = c.CUSTOMER_ID
JOIN (
    SELECT MERCHANT_ID, AVG(AMOUNT) AS MERCH_AVG_TRANSACTION_AMT,
           COUNT(*) AS MERCH_TOTAL_TRANSACTIONS, AVG(IS_FRAUD) AS MERCH_FRAUD_RATE
    FROM RAW_TRANSACTIONS GROUP BY MERCHANT_ID
) m ON t.MERCHANT_ID = m.MERCHANT_ID
LIMIT 10
"""

print("=== SQL-Based Inference ===")
session.sql(sql_query).show()

## Real-Time Inference via HTTP Endpoint

The service exposes a public HTTP endpoint for external applications.

In [ ]:
# Get the endpoint URL
endpoints = session.sql(f"SHOW ENDPOINTS IN SERVICE {SERVICE_NAME}").collect()
for ep in endpoints:
    print(f"Endpoint: {ep}")

print("\n=== HTTP Endpoint Example ===")
print("""
# Call from any external application using HTTP POST:

import requests

URL = 'https://<service-endpoint>.snowflakecomputing.app/predict'
headers = {'Authorization': 'Snowflake Token="<your-pat-token>"'}

# Data format follows Snowflake remote service conventions
data = {
    "data": [
        [0, 5200.0, 1, 3, 2, 0, 1, 8.5, 1,
         150.0, 245, 120.0, 800.0, 35, 0.3, 35, 75000,
         200.0, 1500, 0.01]
    ]
}

response = requests.post(URL, json=data, headers=headers)
print(response.json())
""")

## Service Management

In [ ]:
# Service management commands
print("=== Service Management ===")
print(f"Service: {SERVICE_NAME}")
print(f"Compute Pool: {COMPUTE_POOL}")
print()
print("Commands (uncomment to execute):")
print()
print("# Suspend service (saves compute costs)")
print(f"# session.sql('ALTER SERVICE {SERVICE_NAME} SUSPEND').collect()")
print()
print("# Resume service")
print(f"# session.sql('ALTER SERVICE {SERVICE_NAME} RESUME').collect()")
print()
print("# Delete service when done")
print(f"# mv.delete_service('{SERVICE_NAME}')")
print()
print("Note: With min_instances=0, the service auto-suspends after 30 min of inactivity.")

## Summary

This demo showcased the complete Snowflake ML lifecycle for banking fraud detection:

| Notebook | Capability | What We Did |
|---|---|---|
| 00 | **Data Setup** | Generated 100K synthetic transactions |
| 01 | **Feature Store** | Created Entities, Feature Views (Dynamic Tables), versioned datasets |
| 02 | **Experiment Tracking** | Trained 3 models with autologged metrics, compared runs |
| 03 | **Model Registry** | Logged models with versions, batch inference, SHAP explainability |
| 04 | **Model Serving** | Deployed to SPCS with Python, SQL, and HTTP inference |

### Next: Streamlit App

See the **Streamlit in Snowflake** app (`streamlit/app.py`) for an interactive end-to-end dashboard
that ties all these capabilities together in a single UI.